# 11d Inventory in other seasons

All our stock results so far come from one half-year, December 2015 to May 2016, which includes Christmas. Maybe
that window is unusual. Here we repeat the comparison in two earlier half-years, each with its own warm-up origin and
its own detector threshold, set the same way on data before that window. A product needs 60 full weeks of history
before the window's warm-up to take part.

| window | scored days | covers |
|---|---|---|
| W1 | 2015-12-07 to 2016-05-22 | winter and spring, with Christmas (the original window) |
| W2 | 2015-06-22 to 2015-12-06 | summer and autumn, with Thanksgiving |
| W3 | 2015-01-05 to 2015-06-21 | winter and spring, without Christmas |

LightGBM static, LightGBM retrained and seasonal naive, a 7-day lead time and the base costs, on a finer grid of
targets from 50% to 99.5%.

In [1]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd

from break_detection import to_weekly, weekly_ca_prices
from pipeline import (calibrate_h, intermittency, make_origins, price_daily_frame, run_backtest,
                      simulate_inventory, summarise_inventory)

SEED = 42
L = 7
SLS = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.975, 0.99, 0.995]
POL = [('lightgbm', 'static'), ('lightgbm', 'adaptive'), ('seasonal_naive', 'static')]
HOLDING, STOCKOUT = [0.15, 0.25, 0.35], [0.25, 0.5, 1.0, 2.0]
WINDOWS = {'W1': '2016-05-22', 'W2': '2015-12-06', 'W3': '2015-06-21'}
SEASON = {'W1': 'Dec-May, includes Christmas', 'W2': 'Jun-Dec, includes Thanksgiving', 'W3': 'Jan-Jun, no Christmas'}

long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']], ignore_index=True).sort_values(['item_id', 'date'])
daily = {i: g[['date', 'sales']].reset_index(drop=True) for i, g in long_df.groupby('item_id')}
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
prices = pd.read_csv("../data/raw/sell_prices.csv", dtype={'store_id': 'category'})
price_wk = weekly_ca_prices(prices, calendar, long_df['item_id'].unique())
del prices
saved = pd.read_parquet("../data/processed/backtest_forecasts.parquet")

In [2]:
fcs, meta = {}, []
for w, end in WINDOWS.items():
    origins = make_origins(end, 7)
    eligible = [i for i, d in daily.items() if len(to_weekly(d[d['date'] <= origins[0]])) >= 60]
    if w == 'W1':
        h = None                         # forecasts already exist (09a, threshold calibrated before 2015-11-08)
        fc = saved[saved['item_id'].isin(eligible) & saved['model'].isin(['lightgbm', 'seasonal_naive'])]
    else:
        sub = long_df[long_df['item_id'].isin(eligible)]
        h = calibrate_h({i: daily[i] for i in eligible}, origins[0])
        fc, _, _ = run_backtest(sub, calendar, price_wk, origins, h)
        fc.to_parquet(f"../data/processed/backtest_forecasts_{w}.parquet", index=False)
    fcs[w] = (fc, origins, eligible)
    meta.append({'window': w, 'seasonal_coverage': SEASON[w], 'warm_up_origin': origins[0].date(),
                 'first_eval_day': (origins[1] + pd.Timedelta(days=1)).date(), 'last_eval_day': pd.Timestamp(end).date(),
                 'n_items': len(eligible), 'cusum_h': h})
meta = pd.DataFrame(meta)
meta.to_csv("../outputs/tables/robustness_windows.csv", index=False)
print(meta.to_string(index=False))

local models: 140728 rows


LightGBM origin 2015-05-24 done


LightGBM origin 2015-06-21 done


LightGBM origin 2015-07-19 done


LightGBM origin 2015-08-16 done


...

LightGBM origin 2015-03-01 done


LightGBM origin 2015-03-29 done


LightGBM origin 2015-04-26 done


LightGBM origin 2015-05-24 done


window              seasonal_coverage warm_up_origin first_eval_day last_eval_day  n_items   cusum_h
    W1    Dec-May, includes Christmas     2015-11-08     2015-12-07    2016-05-22      723       NaN
    W2 Jun-Dec, includes Thanksgiving     2015-05-24     2015-06-22    2015-12-06      718 12.809212
    W3          Jan-Jun, no Christmas     2014-12-07     2015-01-05    2015-06-21      709 12.314226


In [3]:
kp = []
for w, (fc, origins, eligible) in fcs.items():
    days = pd.date_range(origins[1] + pd.Timedelta(days=1), periods=28 * 6)
    inter = intermittency(long_df, origins[0])     # classified on data before this window's warm-up origin
    k = simulate_inventory(fc, origins, price_daily_frame(price_wk, days), POL, SLS, L, inter)
    kp.append(k.assign(window=w))
    print(w, len(k), "item simulations")
kpis = pd.concat(kp, ignore_index=True)
kpis.to_csv("../data/processed/robustness_inventory_items.csv", index=False)
summ = summarise_inventory(kpis, by=('window', 'policy', 'service_level'))
summ.to_csv("../outputs/tables/robustness_inventory_summary.csv", index=False)
pd.set_option('display.width', 230)
show = summ[summ['service_level'].isin([0.90, 0.95, 0.99])]
print(show[['window', 'policy', 'service_level', 'n_items', 'fill_rate', 'cycle_service_level', 'stockout_days',
            'avg_inventory_value', 'total_cost']].round(4).to_string(index=False))

# check: W1 must reproduce 11b product by product (same forecasts, same rules)
ref = pd.read_csv("../data/processed/inventory_item_kpis.csv")      # the simulation saved by 11b
ref = ref[(ref['lead_time'] == L)][['item_id', 'policy', 'service_level', 'total_cost']]
w1 = kpis[(kpis['window'] == 'W1') & kpis['service_level'].isin([0.90, 0.95, 0.99])]
m = w1.merge(ref, on=['item_id', 'policy', 'service_level'], suffixes=('', '_nb03'))
print(f"W1 vs 11b: {len(m)} item-policy-level rows, max |total cost difference| = "
      f"{(m['total_cost'] - m['total_cost_nb03']).abs().max():.4f}")

W1 21690 item simulations


W2 21540 item simulations


W3 21270 item simulations


window                policy  service_level  n_items  fill_rate  cycle_service_level  stockout_days  avg_inventory_value  total_cost
    W1     lightgbm_adaptive           0.90      723     0.9665               0.7715           3987          149763.8227  90321.7012
    W1     lightgbm_adaptive           0.95      723     0.9738               0.8189           3048          160157.8204  76123.3422
    W1     lightgbm_adaptive           0.99      723     0.9823               0.8816           1991          180164.3067  58930.2070
    W1       lightgbm_static           0.90      723     0.9705               0.7792           3741          149531.0924  80510.5108
    W1       lightgbm_static           0.95      723     0.9771               0.8277           2828          160085.3247  67630.8484
    W1       lightgbm_static           0.99      723     0.9849               0.8902           1796          179959.7407  52836.4473
    W1 seasonal_naive_static           0.90      723     0.9668      

## Cost differences, with bootstrap intervals over products

In [4]:
rng = np.random.default_rng(SEED)
rows = []
for (w, sl), g in kpis.groupby(['window', 'service_level']):
    t = g.pivot(index='item_id', columns='policy', values='total_cost')
    for label, a, b in [('adaptive - static', 'lightgbm_adaptive', 'lightgbm_static'),
                        ('static - naive', 'lightgbm_static', 'seasonal_naive_static')]:
        d = (t[a] - t[b]).dropna().to_numpy()
        idx = rng.integers(0, len(d), size=(5000, len(d)))
        boot = d[idx].sum(axis=1)
        absd = np.sort(np.abs(d))[::-1]
        rows.append({'window': w, 'service_level': sl, 'comparison': label, 'n_items': len(d),
                     'total_diff': d.sum(), 'total_diff_pct': 100 * d.sum() / t[b].sum(),
                     'ci_low': np.quantile(boot, 0.025), 'ci_high': np.quantile(boot, 0.975),
                     'median_item_diff': np.median(d), 'share_items_lower': np.mean(d < 0),
                     'top10_share_of_abs_gap': absd[:10].sum() / absd.sum()})
diffs = pd.DataFrame(rows)
diffs.to_csv("../outputs/tables/robustness_cost_differences.csv", index=False)
print(diffs[diffs['service_level'].isin([0.90, 0.95, 0.99])].round(4).to_string(index=False))

window  service_level        comparison  n_items  total_diff  total_diff_pct      ci_low    ci_high  median_item_diff  share_items_lower  top10_share_of_abs_gap
    W1           0.90 adaptive - static      723   9811.1904         12.1862   4067.2652 16939.8213            0.0000             0.3167                  0.3883
    W1           0.90    static - naive      723  -6116.8004         -7.0611 -14916.1426  2172.0135           -0.3957             0.5118                  0.2395
    W1           0.95 adaptive - static      723   8492.4938         12.5571   3625.2426 14325.2029            0.0000             0.3264                  0.3928
    W1           0.95    static - naive      723  -8298.4240        -10.9291 -16847.0456  -357.3795           -1.3104             0.5505                  0.2654
    W1           0.99 adaptive - static      723   6093.7596         11.5333   2533.7805 10122.0094            0.0000             0.3057                  0.4340
    W1           0.99    static - 

## Does the textbook optimum match the simulation?

The classic newsvendor answer says the best service level is m / (m + h × 7/365), where m is the cost of a lost sale
and h the yearly holding rate (both as a share of price). We compare it with the cheapest target in the simulation.

In [5]:
rows, be = [], []
for (w, policy), g in kpis.groupby(['window', 'policy']):
    tot = g.groupby('service_level')[['holding_value_days', 'lost_value']].sum()
    for h in HOLDING:
        for m in STOCKOUT:
            cost = h * tot['holding_value_days'] + m * tot['lost_value']
            cr = m / (m + h * 7 / 365)
            grid = np.array(SLS)
            rows.append({'window': w, 'policy': policy, 'holding_rate': h, 'stockout_multiple': m,
                         'critical_ratio': cr, 'nearest_grid_level_to_cr': grid[np.abs(grid - cr).argmin()],
                         'simulated_cheapest_level': cost.idxmin(), 'cost_at_cheapest': cost.min(),
                         'cost_at_99': cost.loc[0.99], 'cost_at_95': cost.loc[0.95]})
    dH = tot.loc[0.99, 'holding_value_days'] - tot.loc[0.95, 'holding_value_days']
    dS = tot.loc[0.95, 'lost_value'] - tot.loc[0.99, 'lost_value']
    be.append({'window': w, 'policy': policy, 'extra_holding_base_99_vs_95': dH, 'saved_stockout_base_99_vs_95': dS,
               'breakeven_stockout_to_holding_ratio': dH / dS if dS > 0 else np.nan})
nv = pd.DataFrame(rows)
nv.to_csv("../outputs/tables/robustness_newsvendor.csv", index=False)
bev = pd.DataFrame(be)
bev.to_csv("../outputs/tables/robustness_breakeven.csv", index=False)
print(nv[nv['policy'] == 'lightgbm_static'][['window', 'holding_rate', 'stockout_multiple', 'critical_ratio',
      'nearest_grid_level_to_cr', 'simulated_cheapest_level']].round(4).to_string(index=False))
print()
print(bev.round(4).to_string(index=False))
print("\nInterpretation: 99% is cheaper than 95% whenever stockout multiple / annual holding rate exceeds the ratio above "
      "(e.g. base case 0.5 / 0.25 = 2).")

window  holding_rate  stockout_multiple  critical_ratio  nearest_grid_level_to_cr  simulated_cheapest_level
    W1          0.15               0.25          0.9886                     0.990                     0.995
    W1          0.15               0.50          0.9943                     0.995                     0.995
    W1          0.15               1.00          0.9971                     0.995                     0.995
    W1          0.15               2.00          0.9986                     0.995                     0.995
    W1          0.25               0.25          0.9812                     0.975                     0.995
    W1          0.25               0.50          0.9905                     0.990                     0.995
    W1          0.25               1.00          0.9952                     0.995                     0.995
    W1          0.25               2.00          0.9976                     0.995                     0.995
    W1          0.35        

## What we found

The results hold in all three windows. W1 reproduces 11b exactly for every product. Static LightGBM is cheaper than
seasonal naive in every window, with the whole 95% interval below zero: by 10.9% in W1, 13.6% in W2 and 13.5% in W3.
Retraining after shifts costs more in every window: 12.6%, 9.1% and 6.7%.

The cost gap between retrained and static is spread out: the ten products with the largest gaps carry 34-39% of it,
not most of it. The textbook optimum and the simulation agree that a very high target is right: the critical ratio is
between 97.4% and 99.9% in every cost scenario, and the simulation's cheapest target was 99.5%, the top of the grid
we tried. A 99% target beats 95% whenever a lost sale costs more than about a quarter of the yearly holding rate.

## What comes next

`11e_classical_forecasts_and_stock` puts the classical models from 07a into the same simulation.

In [6]:
# files this notebook wrote
from pathlib import Path
ROOT = Path('..').resolve()
written = [ROOT / p for p in ['outputs/tables/robustness_windows.csv', 'outputs/tables/robustness_inventory_summary.csv', 'outputs/tables/robustness_cost_differences.csv', 'outputs/tables/robustness_newsvendor.csv', 'outputs/tables/robustness_breakeven.csv']]
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\robustness_windows.csv 0 KB
outputs\tables\robustness_inventory_summary.csv 20 KB
outputs\tables\robustness_cost_differences.csv 9 KB
outputs\tables\robustness_newsvendor.csv 13 KB
outputs\tables\robustness_breakeven.csv 1 KB
